You have characterized and analyzed the sound propagation in the previous practical. We will now
exploit theses properties to infer one sound source position w.r.t.\ a linear microphone array made
of $N=8$ omnidirectional MEMS microphones. The system you will be using is the same as before;
thus, most of the code you already wrote to acquire signals, plot them, etc.\ will remain the same.

In all the following, we will work with a sampling frequency $F_s = 20$kHz, and with a buffer of
size $\texttt{BLK} = 2048$.

In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
#from matplotlib.animation import FuncAnimation
#import ipywidgets as widgets
import numpy as np
from client import array
import time
from IPython.display import display, clear_output

# For autoreloading while developing ;)
%load_ext autoreload
%autoreload 2

# Regarder ça : https://github.com/lvwerra/jupyterplot

In [2]:
antenne=array('server')       # When performing real-time acquisition
#antenne=array('play')  # When playing recorded files

/Users/brunogas/Documents/Dev/Bimea/megamicros/venv/lib/python3.10/site-packages/ipywidgets/widgets/widget_button.py:98: RuntimeWarning: coroutine 'array.validateAcq_fct' was never awaited
  self._click_handlers(self)


In [ ]:
# Load acquisition and array parameters from the antenne variable, after launching acquisition or play
Fs = antenne.fs
BLK = antenne.blocksize
N = antenne.mems_nb
d = antenne.interspace

### 1) To begin, start the acquisition of the audio system, and capture one audio buffer. Plot the resulting signals as a function of time.

In [ ]:
# Lecture d'un buffer audio
m = antenne.read()
print(np.shape(m))

# On crée le vecteur des indices
n = np.arange(0,BLK)
print(m[0,:]/antenne.mu32.sensibility)

# Tracé
fig,ax = plt.subplots()
lines = ax.plot(n,m)
ax.legend(lines, [str(j) for j in range(len(lines))])
plt.xlabel('Sample number')
plt.ylabel('Microphone signals')
plt.title('Microphones signals as a function of time')
plt.show()

### 2) Write the position $z_n$ as a function of $n$ and interspace $d$. As a convention, the first microphone number is selected as $0$.

On peut écrire directement :
$$z_n = \left(n - \frac{N-1}{2}\right) d.$$

### 3) Propose a function $\texttt{beam_filter}$ returning the filter frequency response for one microphone number $\texttt{mic_nb}$. 

In [ ]:
def beam_filter(array, freq_vector, theta0=0, mic_nb: int = 0):
    """Compute the filter frequency response of a DSB beamformer for one microphone

    Args:
        array (array_server obj): array structure controlling the acquisition system.
        freq_vector (np.array): frequency vector. 
        theta0 (int, optional): focusing angular direction (in degrees). Defaults to 0.
        mic_id (int, optional): microphone id. Defaults to 0.

    Returns:
        np.array: the filter frequency response. Shape is (len(freq_vector),).
    """

    N = antenne.mems_nb
    d = antenne.interspace
    # Microphone position z
    z = (mic_nb - (N - 1) / 2) * d
    # Filter's frequency response
    return np.exp(-1j * 2 * np.pi * freq_vector / 340 * z * np.cos(theta0*np.pi/180))

### 4) Plot the two frequency responses obtained for two filters associated to two different microphone outputs when $\theta_0=0^\circ$ and for frequencies between $0$ and $5$kHz. Explain the effect of these filters on the signals.

In [ ]:
f = np.arange(0,5000,10)
theta0 = 0
W0 = beam_filter(antenne,f,theta0,mic_nb=0)
W5 = beam_filter(antenne,f,theta0,mic_nb=5)
fig,axs = plt.subplots(2,sharex=True)
axs[0].plot(f,np.abs(W0))
axs[0].plot(f,np.abs(W5))
axs[1].plot(f,np.angle(W0))
axs[1].plot(f,np.angle(W5))
fig.suptitle('Réponse en fréquence')
plt.xlabel('Fréquence (Hz)')

Comme prévu, les 2 filtres présentent un gain de 1 sur tout l'axe des fréquences. Ces filtres ne diffèrent que de leur phase, qu'on voit linéaire. C'est un comportement attendu compte-tenu de leur expression, qui fait apparaitre une réponse en fréquence du type $e^{-j2\pi f\tau_n}$, donc la phase s'exprime selon $\phi(f) = -2\pi \tau_n f$. Ces filtres introdusent donc un retard pur de valeur $\tau_n$.

### 5) Compare again the filters obtained when $\theta_0 = 90^\circ$. Explain the differences.

In [ ]:
# Pour la position theta0=90
theta0 = 90
W0 = beam_filter(antenne,f,theta0,mic_nb=0)
W5 = beam_filter(antenne,f,theta0,mic_nb=5)
fig,axs = plt.subplots(2,sharex=True)
axs[0].plot(f,np.abs(W0))
axs[0].plot(f,np.abs(W5))
axs[1].plot(f,np.angle(W0))
axs[1].plot(f,np.angle(W5))
fig.suptitle('Réponse en fréquence')
plt.xlabel('Fréquence (Hz)')

On récupère toujours des filtres qui présentent un gain de 1, mais cette fois la phase est (quasi) nulle partout. C'est attendu, car en $\theta_0 = 90^\circ$, on voit que tous les filtres sont égaux et valent $W(f)=1$. Dans cette situation, l'antenne est polarisée dans un direction où les signaux de sortie des microphones sont en phase lorsque la source est en face, i.e. lorsque tous les signaux sont identiques entre eux (hypothèse de champ lointain). En conséquence, la formation de voie "se contente" de sommer les signaux, i.e. les filtres n'ont "aucun" effet de retard sur les signaux mesurés.

### 6) After acquiring an audio buffer, compute its FFT. Plot the result of this analysis as a function of the frequency when emitting a pure sine tone with a frequency $F_0=1$kHz.

In [ ]:
# On fait l'acquisition d'une trame
m = antenne.read()
# Calcul de la FFT
M_fft = np.fft.fft(m,axis=0)
# Vecteur en fréquence correspondant
f_fft = np.arange(0, Fs, Fs/BLK)
# Tracé
plt.figure()
plt.plot(f_fft.T,np.abs(M_fft))
plt.xlim(0,2000) 

Comme prévu, on mesure un pic de la FFT à la fréquence $f=1$kHz. Les amplitudes sont aussi différentes d'un microphone à l'autre ... alors qu'on avait dit qu'il devait y en avait aucune. Les réverbérations sont certainement à l'origine de cette différence, sans que cela ne soit un soucis sur l'analyse qui en sera faite ensuite.

### 7) Among all the frequencies you obtained from the FFT, select the one corresponding to the source frequency. Give its exact value, and collect the corresponding FFT values in one vector $\texttt{M}$ of length $N$.

In [ ]:
# Position du max de la FFT du micro 1
idx = np.argmax(np.abs(M_fft[:,1]))
# Valeur de la fréquence et vecteur M associé
f = f_fft[idx]
M = M_fft[idx,:]

print('Maximal value is at',f,'Hz')
print(M)

### 8) In a loop along all microphones, compute each filters for the position $\theta_0$ and the frequency value you obtained in the previous step. Apply then these filters to the microphones outputs FFT, restricted to the same frequency.

In [ ]:
theta0 = 180
filter_outputs = np.zeros(N, dtype=np.complex_)
for idx_mic in np.arange(0, N):
    # Compute the associated filter
    W = beam_filter(antenne, f, theta0, idx_mic)
    # Apply it to the FFT of each microphones
    filter_outputs[idx_mic] = M[idx_mic] * W
    
print(np.size(filter_outputs))

### 9) Combine then the filters outputs to form the beamformer output $Y_{\theta_0}[k_0]$. *$Y_{\theta_0}[k_0]$ is obviously a complex value which corresponds to the frequency contribution of the source to the $k_0^{\text{th}}$ frequency component of the beamformer output when focalized in the direction $\theta_0$.* Compute then the corresponding power $P(\theta_0)$ at $k_0$ of the beamformer output.

In [ ]:
# The beamformer output is the sum of the filters outputs
Y = np.sum(filter_outputs)
# The correspoding power at the frequency F0 is then abs(Y)^2
P = np.abs(Y)**2
print(P)

### 10) For a direction $\theta_0$ of your choice, compute $P(\theta_0)$ for (i) a source emitting from a direction close to $\theta_0$, or (ii) far from it. Compare the two values.

In [ ]:
# Pour theta0 = 90°
theta0 = 90
filter_outputs = np.zeros(N, dtype=np.complex_)
for idx_mic in np.arange(0, N):
    W = beam_filter(antenne, f, theta0, idx_mic)
    filter_outputs[idx_mic] = M[idx_mic] * W
Y = np.sum(filter_outputs)
P = np.abs(Y)**2
print(P)

In [ ]:
# Pour theta0 = 0°
theta0 = 0
filter_outputs = np.zeros(N, dtype=np.complex_)
for idx_mic in np.arange(0, N):
    W = beam_filter(antenne, f, theta0, idx_mic)
    filter_outputs[idx_mic] = M[idx_mic] * W
Y = np.sum(filter_outputs)
P = np.abs(Y)**2
print(P)

La source émettait "vaguement" depuis en face. La puissance apparait plus forte pour la position 90° que pour la position 0°, comme attendu.

### 11) Repeat now the previous code in a loop for $\theta_0$ values ranging from $0$ to $180^{\circ}$. You should then obtain an array $P$ where each value corresponds to the power of the beamformer output at $F_0$ for each angular polarization. Plot the array $P$ as a function of the angle $\theta_0$.

In [ ]:
theta0 = np.arange(0,181)
filter_outputs = np.zeros(N, dtype=np.complex_)
P = np.zeros(len(theta0))
for idx, angle in enumerate(theta0):
  for idx_mic in np.arange(0, N):
    W = beam_filter(antenne, f, angle, idx_mic)
    filter_outputs[idx_mic] = M[idx_mic] * W
  Y = np.sum(filter_outputs)
  P[idx] = np.abs(Y)**2

# Tracé
plt.figure()
plt.plot(theta0,P)
#idx = np.argmax(P)
#plt.bar(theta0[idx],height=np.max(P)*1.05,color='red')


### 12) Find the $\theta_0$ value corresponding to position of the maximum in P and compare it with the actual (but approximate) position of the sound source.

In [ ]:
# Position du max de P
idx = np.argmax(P)
print("La position estimée de la source est theta =",theta0[idx],'degrés.')

## 3) Analyzing the beamformer performances

In [ ]:
from beamformer import beamformer

theta0 = np.arange(0, 180, 1)
F0 = 1000

# On fait l'acquisition d'une trame
m = antenne.read()
P = beamformer(m,theta0,F0,Fs)

# Tracé
figure, ax = plt.subplots()
line1, = ax.plot(theta0, P)
#line1, = ax.plot(m[2,:].T)
idx = np.argmax(P)
maxi = np.max(P)
ax.set_xlim(0, 180)
ax.set_ylim(0, maxi*1.05)

In [ ]:
i=0
Pmax = np.zeros(10000)
try:
    while True:
        # Read microphones data
        m = antenne.read()

        # Compute beamforming
        P = beamformer(m,theta0,F0,Fs)

        idx = np.argmax(P)
        if (np.max(P)>maxi):
            maxi = np.max(P)
        #else:
        #    maxi = 0.8*maxi

        line1.set_ydata(P)
        #line1.set_ydata(m[2,:].T)
        ax.set_xlim(0, 180)
        ax.set_ylim(0, maxi)
        figure.canvas.draw()
        figure.canvas.flush_events()
        plt.pause(0.1)
        
        clear_output(wait=True)
        display('La position de la source est = '+str(theta0[idx])+'degrés.')
        Pmax[i]=theta0[idx]
        i=i+1
        
        time.sleep(0.15)

except KeyboardInterrupt:
    pass

In [ ]:
plt.figure()
plt.plot(Pmax[0:i])

## 2) Générer une source sinusoidale d'une fréquence autour de $500$ Hz depuis 2 positions par rapport à l'antenne (idéalement : en face, puis sur le côté), relativement proche de celle-ci. Vérifier qualitativement les caractéristiques attendues de la propagation (atténation, retard). Si le résultat n'est pas celui attendu, expliquer.

In [ ]:


# On trace les 8 signaux (ou seulement ceux issus des micros 1 et 8 ?)
# - pour une source en face : les signaux sont "à peu prêt en phase"
# - depuis une direction latérale : les signaux se déphasent entre eux
# 

## 3) Soit une source sinusoidale de fréquence $500$Hz. Mesurer le décalage $\Delta T_{18}$ entre les 2 signaux $m_1(t)$ et $m_2(t)$. En déduire, par un simple calcul, la position estimée correspondante.

In [ ]:
# Tracé uniquement pour les micros 1 et 8

plt.figure()
plt.plot(n,m[0,:])
plt.plot(n,m[7,:])
plt.show()

# On mesure N échantillons de décalages
N = 10
# On estime la position alors par
print('theta =',np.arccos(N*1/antenne.ACQ['fs']*antenne.ARRAY['snd_speed']/antenne.ARRAY['interspace']/(antenne.ARRAY['mems_nb']))*180/np.pi)

# Commenter la précision que l'on peut obtenir en terme de position angulaire
# theta = acos(tau*c/d) et tau = k/Fs => theta = acos(k*c/(d*Fs))
# Le plus grand retard est atteint quand theta = 0° ou 180°, alors tau = d/c = 176,5 µs
# Or avec Fs = 10kHz, Ts = 1/Fs = 100µs ... donc soit k=0 => tau=0, soit k=1 => tau = Te => theta = 55.48° !
# Et si tau = 2Ts, alors tau > d/C => impossible en théorie. Donc que 2 angles possibles : 0 ou 55°
# => seul moyen : augmenter la fréquence d'échantillonnage
# Pour Fs = 50kHz, alors : 90°; 83.5°; 76.9°; 70.1° ; 63° ; 55.5° ; 47.2° ; 37.5° ; 25° et c'est tout (8 positions possibles)

## 4) Dans le cas général, exprimer l'angle estimé de la position de la sourcec sonore en fonction de $\Delta T_{ij}$. Sachant que les retards estimés ne peuvent être que multiple de la période d'échantillonnage, quelles positions peuvent être estimées pour une fréquence d'échantillonnage $F_s = 10$kHz ?